# ViT5 inference trên 100 bài Vietnews test

Settings Kaggle: GPU T4 x2, Internet On, Private.
Model: `VietAI/vit5-base-vietnews-summarization`
Theo model card: không prefix, thêm `</s>`, `max_length=256`.
Code dùng 1 GPU (`cuda:0`). Upload thư mục `data/test_100` hoặc `test_ids` + file `.seg` thành Dataset.

In [ ]:
!nvidia-smi
!pip -q install transformers sentencepiece

In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "VietAI/vit5-base-vietnews-summarization"
DATA = Path("/kaggle/input")  # chỉnh sau khi add dataset
# Tìm thư mục chứa *.txt.seg
cands = list(DATA.rglob("*.txt.seg"))
print("n files", len(cands))
files = sorted(cands)[:100]
print("first", files[0] if files else None)

In [ ]:
def parse_file(path):
    parts = [p.strip() for p in Path(path).read_text(encoding="utf-8").split("\n\n") if p.strip()]
    return {"id": Path(path).name, "title": parts[0], "abstract": parts[1], "body": "\n".join(parts[2:])}

docs = [parse_file(p) for p in files]
print(docs[0]["id"], len(docs[0]["body"].split()))

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device", device)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)
model.eval()

# test 1 bài
sample = docs[0]["body"] + "</s>"
enc = tok(sample, return_tensors="pt", truncation=True, max_length=1024)
enc = {k: v.to(device) for k, v in enc.items()}
with torch.no_grad():
    out = model.generate(**enc, max_length=256, early_stopping=True)
print(tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True))

In [ ]:
preds = []
for i, doc in enumerate(docs):
    text = doc["body"] + "</s>"
    enc = tok(text, return_tensors="pt", truncation=True, max_length=1024)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_length=256, early_stopping=True)
    pred = tok.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    preds.append({"id": doc["id"], "abstract": doc["abstract"], "pred": pred})
    if (i + 1) % 10 == 0:
        print("done", i + 1)

out_path = Path("/kaggle/working/preds.json")
out_path.write_text(json.dumps(preds, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", out_path, "n=", len(preds))